# Sesión 07 — Lab CDF: Change Data Feed en Delta Lake
## Captura y Propagación Incremental de Cambios (Bronze → Silver)

**Curso:** Databricks Data Engineer Associate
**Runtime mínimo:** DBR 13.3 LTS
**Plataforma:** Azure Databricks con Unity Catalog habilitado
**Dataset:** `suscripciones_usuarios.csv`

### Objetivos del laboratorio
1. Entender **qué es** Change Data Feed (CDF) y por qué resuelve un problema real de los pipelines incrementales.
2. Habilitar CDF en una tabla Bronze de Unity Catalog.
3. Comprender los **4 tipos de cambio** que produce el CDF: `insert`, `update_preimage`, `update_postimage`, `delete`.
4. Leer el CDF con la API de DataFrame **y** con la función SQL `table_changes()`.
5. Propagar cambios incrementalmente de Bronze a Silver con `MERGE INTO`.
6. Implementar un **patrón de watermark** (rastreo de versión) para garantizar idempotencia.
7. Ejecutar una **segunda ronda de cambios** y demostrar que el pipeline solo procesa lo nuevo, sin reprocesar nada — la prueba real de que el pipeline es incremental.

## Marco conceptual — ¿Qué es Change Data Feed?

Por defecto, una tabla Delta solo te deja ver su **estado actual** (o un estado pasado completo con time travel). Si quieres saber *qué cambió* entre la versión 5 y la versión 9, tendrías que comparar ambas tablas completas — costoso e ineficiente, sobre todo en tablas grandes.

**Change Data Feed (CDF)** resuelve esto: cuando está habilitado, Delta Lake registra automáticamente **cada fila afectada por cada operación de escritura** (`INSERT`, `UPDATE`, `DELETE`, `MERGE`), junto con metadata de auditoría. Esto permite leer únicamente *lo que cambió*, en lugar de recalcular o comparar la tabla completa.

### Los 4 tipos de cambio (`_change_type`)

| Tipo | Cuándo se genera | Qué representa | Uso típico en el pipeline |
|---|---|---|---|
| `insert` | Una fila nueva se agrega (carga inicial, `INSERT`, o `MERGE ... WHEN NOT MATCHED`) | El estado final de la fila nueva | Propagar como `INSERT` en Silver |
| `update_preimage` | Una fila existente se modifica | El valor **antes** del `UPDATE` | Solo para auditoría / trazabilidad histórica — **no** se propaga |
| `update_postimage` | Una fila existente se modifica | El valor **después** del `UPDATE` | Propagar como `UPSERT` en Silver |
| `delete` | Una fila se elimina (`DELETE` o `MERGE ... WHEN MATCHED THEN DELETE`) | El último valor de la fila antes de borrarse | Propagar como `DELETE` en Silver |

Cada registro del CDF además incluye tres columnas de metadata:

- **`_commit_version`**: la versión del Delta log en la que ocurrió el cambio.
- **`_commit_timestamp`**: el timestamp del commit.
- **`_change_type`**: uno de los cuatro valores de la tabla anterior.

> **Punto clave:** CDF **no es retroactivo**. Debe habilitarse antes de que ocurran los cambios que se quieren capturar; si se activa sobre una tabla existente, solo capturará cambios **a partir de ese momento**, no el historial previo.

> **Limitación a tener en cuenta:** los datos del CDF dependen de los archivos físicos subyacentes. Si se ejecuta `VACUUM` y se eliminan archivos de versiones antiguas, el CDF de esas versiones deja de estar disponible (igual que con time travel). En producción, la retención del log/archivos (`delta.logRetentionDuration`, `delta.deletedFileRetentionDuration`) debe ser mayor a la frecuencia con la que se ejecuta el pipeline incremental.

In [0]:
# Configuración — ejecutar esta celda antes de comenzar
CATALOGO      = "dbassociate"
SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"
VOL_LANDING   = "/Volumes/dbassociate/default/vol_landing/sesion_07"
CSV_SUSCS     = f"{VOL_LANDING}/suscripciones_usuarios.csv"

TABLA_BRONZE  = f"{CATALOGO}.{SCHEMA_BRONZE}.suscripciones"
TABLA_SILVER  = f"{CATALOGO}.{SCHEMA_SILVER}.suscripciones"

# Verificar que el archivo existe en el Volume
try:
    dbutils.fs.ls(CSV_SUSCS)
    print(f"OK: {CSV_SUSCS}")
except Exception:
    raise FileNotFoundError(
        f"Subir suscripciones_usuarios.csv a {VOL_LANDING} antes de continuar."
    )

## Paso 1 — Crear la tabla Bronze con CDF habilitado

CDF se habilita con la propiedad de tabla `delta.enableChangeDataFeed = true`. Hay tres formas de activarlo:

1. **Al crear la tabla** (la que usamos aquí), vía `TBLPROPERTIES`.
2. **Sobre una tabla existente**, vía `ALTER TABLE ... SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')` — solo capturará cambios futuros.
3. **A nivel de sesión/clúster**, vía `spark.conf.set("spark.databricks.delta.properties.defaults.enableChangeDataFeed", "true")` — aplica por defecto a toda tabla Delta nueva.

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {TABLA_BRONZE}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLA_BRONZE} (
        suscripcion_id      STRING NOT NULL,
        usuario_id          STRING,
        plan                STRING,
        estado              STRING,
        fecha_inicio        DATE,
        monto_mensual       DECIMAL(8,2),
        canal_adquisicion   STRING,
        pais                STRING
    )
    USING DELTA
    TBLPROPERTIES (
        'delta.enableChangeDataFeed' = 'true'
    )
""")

print(f"Tabla creada: {TABLA_BRONZE}")
print("CDF habilitado: la tabla capturará INSERT, UPDATE y DELETE desde este momento.")

# Alternativa sobre una tabla ya existente (no se ejecuta, solo referencia):
# spark.sql(f"ALTER TABLE {TABLA_BRONZE} SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")

In [0]:
%sql
describe detail dbassociate.bronze.suscripciones

## Paso 2 — Carga inicial desde CSV

La carga inicial queda registrada en el Delta log como **versión 0**, y en el CDF como registros de tipo `insert`.
Para la propagación incremental arrancamos a leer el CDF desde la **versión 1**, porque la versión 0 la cargamos directamente a Silver (no tendría sentido leerla del CDF y volver a insertarla).

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DateType, DecimalType

schema_suscs = StructType([
    StructField("suscripcion_id",    StringType(),       nullable=False),
    StructField("usuario_id",        StringType(),       nullable=True),
    StructField("plan",              StringType(),       nullable=True),
    StructField("estado",            StringType(),       nullable=True),
    StructField("fecha_inicio",      DateType(),         nullable=True),
    StructField("monto_mensual",     DecimalType(8, 2),  nullable=True),
    StructField("canal_adquisicion", StringType(),       nullable=True),
    StructField("pais",              StringType(),       nullable=True),
])

df_suscs = (
    spark.read
    .option("header", "true")
    .schema(schema_suscs)
    .csv(CSV_SUSCS)
)

df_suscs.write.format("delta").mode("append").saveAsTable(TABLA_BRONZE)

print(f"Carga inicial completada: {df_suscs.count()} registros — versión 0 en el delta log")

In [0]:
%sql
describe history dbassociate.bronze.suscripciones


In [0]:
# DESCRIBE HISTORY muestra cada commit del Delta log: útil para correlacionar
# las versiones que vamos a leer del CDF con la operación que las generó.
display(spark.sql(f"DESCRIBE HISTORY {TABLA_BRONZE}").select("version", "timestamp", "operation"))

## Paso 3 — Ronda 1 de cambios de negocio

Simulamos tres operaciones reales sobre Bronze, cada una genera un tipo distinto de registro en el CDF:

| Operación | Regla de negocio | Tipo de cambio generado |
|---|---|---|
| `UPDATE` plan | Upgrade a Premium para suscripciones **Básico** adquiridas por **Referido** | `update_preimage` + `update_postimage` |
| `UPDATE` estado | Suspender suscripciones **Activas** en **Perú** | `update_preimage` + `update_postimage` |
| `DELETE` | Eliminar suscripciones **Canceladas** en **Chile** | `delete` |

Estas reglas fueron verificadas contra el CSV real para asegurar que cada una afecte al menos una fila — de lo contrario el CDF no tendría nada que mostrar para ese tipo de cambio.

In [0]:
# Cambio 1: upgrade de plan para suscripciones Básico adquiridas por Referido
spark.sql(f"""
    UPDATE {TABLA_BRONZE}
    SET plan = 'Premium', monto_mensual = 199.99
    WHERE plan = 'Básico'
    AND canal_adquisicion = 'Referido'
""")
print("Cambio 1: upgrades aplicados")

# Cambio 2: marcar como Suspendido las suscripciones Activas en Perú
spark.sql(f"""
    UPDATE {TABLA_BRONZE}
    SET estado = 'Suspendido'
    WHERE pais = 'Perú'
    AND estado = 'Activo'
""")
print("Cambio 2: suspensiones aplicadas")

# Cambio 3: eliminar suscripciones canceladas en Chile
spark.sql(f"""
    DELETE FROM {TABLA_BRONZE}
    WHERE estado = 'Cancelado'
    AND pais = 'Chile'
""")
print("Cambio 3: cancelaciones eliminadas")

In [0]:
%sql
describe history dbassociate.bronze.suscripciones

## Paso 4 — Leer el Change Data Feed

Hay dos formas equivalentes de leer el CDF: la **API de DataFrame** (con `.option("readChangeFeed", "true")`) y la **función SQL `table_changes(tabla, version_inicio[, version_fin])`**. Ambas son examinables en la certificación Data Engineer Associate; veamos las dos.

In [0]:
# Opción A — API de DataFrame
df_cdf = spark.read.format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 1) \
    .table(TABLA_BRONZE)
    #.option("endingVersion", 2) \
    

print("=== Columnas del CDF (columnas originales + metadata) ===")
print([c for c in df_cdf.columns])
print(f"Total de registros en el CDF: {df_cdf.count()}")
print("=== Distribución por tipo de cambio y versión ===")


display(
    df_cdf.groupBy("_change_type", "_commit_version")
    .count()
    .orderBy("_commit_version", "_change_type")
)

In [0]:
df_cdf.display()

In [0]:
# Opción B — Función SQL table_changes(): mismo resultado, sintaxis SQL pura
display(spark.sql(f"""
    SELECT _change_type, _commit_version, count(*) AS total
    FROM table_changes('{TABLA_BRONZE}', 1)
    GROUP BY _change_type, _commit_version
    ORDER BY _commit_version, _change_type
"""))

In [0]:
%sql
select * 
from table_changes('dbassociate.bronze.suscripciones', 2,2)
order by _commit_version asc, _change_type asc, suscripcion_id asc

## Paso 5 — Filtrar por tipo de cambio

El CDF genera múltiples registros por cada operación. Para propagar correctamente a Silver:
- **`insert`**: nuevos registros → `INSERT` en Silver.
- **`update_postimage`**: estado final tras el `UPDATE` → `UPSERT` en Silver.
- **`update_preimage`**: estado anterior al `UPDATE` → se **ignora** para la propagación (solo sirve para auditoría/trazabilidad histórica).
- **`delete`**: registros eliminados → `DELETE` en Silver.

In [0]:
from pyspark.sql import functions as F

# Mostrar qué generó cada tipo de cambio
for tipo in ["insert", "update_preimage", "update_postimage", "delete"]:
    df_tipo = df_cdf.filter(F.col("_change_type") == tipo)
    
    print(f"\n--- {tipo}: {df_tipo.count()} registros ---")
    
    if df_tipo.count() > 0:
        display(df_tipo.select(
            "_change_type", "_commit_version", "_commit_timestamp",
            "suscripcion_id", "plan", "estado", "monto_mensual"
        ))

## Paso 6 — Crear Silver y encapsular la propagación en una función reutilizable

En lugar de repetir el mismo bloque de `MERGE` cada vez que el pipeline corre, lo encapsulamos en una función `propagar_cdf_a_silver()`. Esto es justamente lo que la convierte en un **pipeline incremental real**: la misma función se invoca en cada ejecución, leyendo siempre desde la última versión procesada.

In [0]:
# Inicializar Silver con el estado de la carga original (versión 0 de Bronze)
df_bronze_v0 = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .table(TABLA_BRONZE)

spark.sql(f"DROP TABLE IF EXISTS {TABLA_SILVER}")
df_bronze_v0.write.format("delta").mode("overwrite").saveAsTable(TABLA_SILVER)

print(f"Silver inicializado con {df_bronze_v0.count()} registros desde la versión 0 de Bronze")

In [0]:
%sql
select * from dbassociate.silver.suscripciones

### Por qué deduplicamos antes del `MERGE`

Dentro de **una misma ventana incremental** (un mismo `desde_version` hasta la última versión disponible), una misma `suscripcion_id` puede aparecer en más de un registro del CDF — por ejemplo, si se actualiza dos veces, o si se inserta y luego se actualiza, antes de que el pipeline vuelva a correr.

Si esas filas se pasan tal cual a un único `MERGE`:
- Si **dos filas fuente coinciden (`MATCHED`)** con la misma fila destino, Delta lanza un error (`multiple source rows matched`).
- Si **dos filas fuente con la misma clave no coinciden (`NOT MATCHED`)**, Delta **no valida unicidad** en el `INSERT` del `MERGE` y **inserta ambas** → filas duplicadas en Silver.

La forma correcta de evitarlo es quedarnos, por cada clave, **solo con el cambio de mayor `_commit_version`** dentro del lote (el estado más reciente "gana"), y aplicar un **único `MERGE` condicional** — no dos merges separados — para que el orden real insert/update/delete del lote se respete (si el último evento de una clave es un `delete`, debe borrarse, sin importar que antes haya tenido un `insert` o `update` en el mismo lote).

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

def propagar_cdf_a_silver(desde_version: int, descripcion: str = "") -> dict:
    """
    Lee el Change Data Feed de TABLA_BRONZE desde 'desde_version' (inclusive),
    deduplica por clave (quedándose con el último cambio dentro del lote) y lo
    propaga a TABLA_SILVER con un único MERGE condicional.

    Retorna un resumen: {'version_final', 'filas_aplicadas'}.
    'version_final' es la última versión de Bronze incluida en este procesamiento
    (se usa para actualizar el watermark).
    """
    df_cdf_run = (
        spark.read.format("delta")
        .option("readChangeFeed", "true")
        .option("startingVersion", desde_version)
        .table(TABLA_BRONZE)
    )

    if df_cdf_run.count() == 0:
        print(f"[{descripcion}] No hay cambios nuevos desde la versión {desde_version}.")
        return {"version_final": desde_version - 1, "filas_aplicadas": 0}

    # Solo nos interesan estos tres tipos para propagar (update_preimage se descarta)
    df_cdf_relevante = df_cdf_run.filter(
        F.col("_change_type").isin(["insert", "update_postimage", "delete"])
    )

    # Deduplicar: nos quedamos con el cambio más reciente por suscripcion_id en el lote
    ventana_latest = Window.partitionBy("suscripcion_id").orderBy(F.col("_commit_version").desc())
    df_cdf_dedup = (
        df_cdf_relevante
        .withColumn("_rn", F.row_number().over(ventana_latest))
        .filter(F.col("_rn") == 1)
        .drop("_rn", "_commit_timestamp")
    )

    n_upserts = df_cdf_dedup.filter(F.col("_change_type") != "delete").count() # insert + update
    n_deletes = df_cdf_dedup.filter(F.col("_change_type") == "delete").count()
    df_cdf_dedup.createOrReplaceTempView("cdf_dedup_tmp")

    # MERGE único y condicional: el _change_type de la fila ya deduplicada decide
    # si corresponde DELETE, UPDATE o INSERT. Se listan columnas explícitamente
    # porque la fuente (cdf_dedup_tmp) trae columnas extra (_change_type, _commit_version)
    # que no existen en la tabla destino, por lo que no se puede usar "UPDATE SET *".
    spark.sql(f"""
        MERGE INTO {TABLA_SILVER} AS target
        USING cdf_dedup_tmp AS source
        ON target.suscripcion_id = source.suscripcion_id
        WHEN MATCHED AND source._change_type = 'delete' THEN
            DELETE
        WHEN MATCHED THEN
            UPDATE SET
                target.usuario_id        = source.usuario_id,
                target.plan              = source.plan,
                target.estado            = source.estado,
                target.fecha_inicio      = source.fecha_inicio,
                target.monto_mensual     = source.monto_mensual,
                target.canal_adquisicion = source.canal_adquisicion,
                target.pais              = source.pais
        WHEN NOT MATCHED AND source._change_type != 'delete' THEN
            INSERT (suscripcion_id, usuario_id, plan, estado, fecha_inicio, monto_mensual, canal_adquisicion, pais)
            VALUES (source.suscripcion_id, source.usuario_id, source.plan, source.estado,
                    source.fecha_inicio, source.monto_mensual, source.canal_adquisicion, source.pais)
    """)

    version_final = df_cdf_run.agg(F.max("_commit_version")).collect()[0][0]

    print(f"[{descripcion}] Procesado: versiones {desde_version} → {version_final}")
    print(f"  → Claves con upsert (insert/update) tras deduplicar: {n_upserts}")
    print(f"  → Claves con delete tras deduplicar: {n_deletes}")

    return {"version_final": version_final, "filas_aplicadas": n_upserts + n_deletes}

In [0]:
# Primera ejecución del pipeline: procesar la Ronda 1 completa (versión 1 en adelante)
resultado_ronda_1 = propagar_cdf_a_silver(desde_version=1, descripcion="Ronda 1")

print(f"\nTotal registros en Silver tras Ronda 1: {spark.table(TABLA_SILVER).count()}")
# display(spark.table(TABLA_SILVER).groupBy("plan", "estado").count().orderBy("plan", "estado"))

## Paso 7 — Watermark: rastreo de versión para garantizar idempotencia

El patrón correcto: después de cada ejecución exitosa del pipeline, guardar la última versión de Bronze procesada en una tabla de control. La siguiente ejecución lee el CDF **a partir de esa versión + 1**, no desde el principio.

> **Sin este rastreo:** cada ejecución reprocesaría todos los cambios desde la versión 0, generando duplicados o re-aplicando operaciones ya hechas (no es idempotente).

In [0]:
resultado_ronda_1

In [0]:
NOMBRE_PIPELINE = "cdf_bronze_to_silver_suscripciones"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOGO}.{SCHEMA_BRONZE}.pipeline_watermarks (
        pipeline_name       STRING,
        tabla_fuente        STRING,
        last_version        LONG,
        updated_at          TIMESTAMP
    )
    USING DELTA
""")

def guardar_watermark(version: int):
    spark.sql(f"""
        MERGE INTO {CATALOGO}.{SCHEMA_BRONZE}.pipeline_watermarks AS target
        USING (
            SELECT
                '{NOMBRE_PIPELINE}' AS pipeline_name,
                '{TABLA_BRONZE}' AS tabla_fuente,
                {version} AS last_version,
                current_timestamp() AS updated_at
        ) AS source
        ON target.pipeline_name = source.pipeline_name
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

def leer_watermark() -> int:
    fila = spark.sql(f"""
        SELECT last_version FROM {CATALOGO}.{SCHEMA_BRONZE}.pipeline_watermarks
        WHERE pipeline_name = '{NOMBRE_PIPELINE}'
    """).collect()
    return fila[0][0] if fila else 0

# Guardar el watermark de la Ronda 1
print(f"Tabla tecnica creada: {CATALOGO}.{SCHEMA_BRONZE}.pipeline_watermarks")

guardar_watermark(resultado_ronda_1["version_final"])

print(f"Watermark guardado: última versión procesada = {leer_watermark()}")

In [0]:
%sql
select * from dbassociate.bronze.pipeline_watermarks

## Paso 8 — Ronda 2: nuevos cambios de negocio (la prueba real de incrementalidad)

En producción el negocio no se detiene: siguen llegando `UPDATE`s y `DELETE`s sobre Bronze. La pregunta que debe responder este laboratorio es: **¿el pipeline vuelve a tocar lo que ya procesó en la Ronda 1, o solo lee lo nuevo?**

Generamos tres cambios de negocio adicionales, independientes de los de la Ronda 1:

| Operación | Regla de negocio | Tipo de cambio |
|---|---|---|
| `UPDATE` precio | Incremento de 10% para plan **Premium** en **Colombia** | `update_preimage` + `update_postimage` |
| `UPDATE` estado | Reactivar (a `Activo`) suscripciones **Suspendidas** en **México** | `update_preimage` + `update_postimage` |
| `DELETE` | Eliminar suscripciones **Estándar** **Canceladas** (depuración de cuentas cerradas) | `delete` |

In [0]:
# Cambio 4: incremento de precio 10% para Premium en Colombia
spark.sql(f"""
    UPDATE {TABLA_BRONZE}
    SET monto_mensual = ROUND(monto_mensual * 1.10, 2)
    WHERE plan = 'Premium'
    AND pais = 'Colombia'
""")
print("Cambio 4: incremento de precio aplicado")

# Cambio 5: reactivar suscripciones suspendidas en México
spark.sql(f"""
    UPDATE {TABLA_BRONZE}
    SET estado = 'Activo'
    WHERE pais = 'México'
    AND estado = 'Suspendido'
""")
print("Cambio 5: reactivaciones aplicadas")

# Cambio 6: depurar cuentas Estándar canceladas
spark.sql(f"""
    DELETE FROM {TABLA_BRONZE}
    WHERE plan = 'Estándar'
    AND estado = 'Cancelado'
""")
print("Cambio 6: cuentas canceladas depuradas")

In [0]:
%sql
describe history dbassociate.bronze.suscripciones

### Verificar que el pipeline lee solo lo nuevo

Antes de propagar, comprobamos explícitamente que el rango de versiones que vamos a leer arranca **después** del watermark de la Ronda 1 — es decir, que no estamos a punto de reprocesar nada.

In [0]:
ultima_version_ronda_1 = leer_watermark()
desde_version_ronda_2 = ultima_version_ronda_1 + 1

print(f"Watermark guardado por la Ronda 1: versión {ultima_version_ronda_1}")
print(f"La Ronda 2 leerá el CDF a partir de la versión: {desde_version_ronda_2}")

In [0]:
# Inspeccionar qué versiones trae el CDF a partir de ese punto
df_cdf_ronda_2 = spark.read.format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", desde_version_ronda_2) \
    .table(TABLA_BRONZE)


versiones_en_ronda_2 = sorted(r[0] for r in df_cdf_ronda_2.select("_commit_version").distinct().collect())
print(f"Versiones presentes en este lote: {versiones_en_ronda_2}")
assert min(versiones_en_ronda_2) > ultima_version_ronda_1, \
    "ERROR: el lote incluye versiones ya procesadas en la Ronda 1"
print("✓ Confirmado: ninguna versión de la Ronda 1 está siendo reprocesada.")

In [0]:
# Ejecutar el mismo pipeline (misma función) para la Ronda 2
resultado_ronda_2 = propagar_cdf_a_silver(desde_version=desde_version_ronda_2, descripcion="Ronda 2")

# Actualizar el watermark
guardar_watermark(resultado_ronda_2["version_final"])
print(f"\nWatermark actualizado: última versión procesada = {leer_watermark()}")

In [0]:
%sql
select * from dbassociate.bronze.pipeline_watermarks

## Paso 9 — Verificación final: consistencia Bronze ↔ Silver

Tras dos rondas de cambios (3 operaciones cada una), Bronze y Silver deben quedar perfectamente sincronizados: mismo número de filas y misma distribución por `plan`/`estado`.

In [0]:
total_bronze = spark.table(TABLA_BRONZE).count()
total_silver = spark.table(TABLA_SILVER).count()

print(f"Total filas en Bronze: {total_bronze}")
print(f"Total filas en Silver: {total_silver}")
print(f"¿Coinciden? {'✓ SÍ' if total_bronze == total_silver else '✗ NO'}")

print("\n=== Distribución Bronze ===")
display(spark.table(TABLA_BRONZE).groupBy("plan", "estado").count().orderBy("plan", "estado"))

print("\n=== Distribución Silver ===")
display(spark.table(TABLA_SILVER).groupBy("plan", "estado").count().orderBy("plan", "estado"))

print("\n=== Historial completo de versiones de Bronze ===")
display(spark.sql(f"DESCRIBE HISTORY {TABLA_BRONZE}").select("version", "timestamp", "operation"))

## Conclusiones clave

1. **CDF captura cambios fila por fila**, no solo "algo cambió" — distingue `insert`, `update_preimage`, `update_postimage` y `delete`, lo que permite propagar exactamente el tipo de operación correcta en destino.
2. **No es retroactivo**: debe habilitarse antes de los cambios que se quieren capturar.
3. **`table_changes()` (SQL) y `.option("readChangeFeed", "true")` (DataFrame API)** son dos formas equivalentes de leer el mismo feed.
4. Un pipeline incremental real necesita **dos piezas**: la lectura del CDF desde una versión, y un **watermark** que recuerde hasta dónde se procesó — sin watermark no hay idempotencia.
5. La Ronda 2 demostró el punto central del laboratorio: al guardar y reutilizar el watermark, el pipeline **nunca vuelve a tocar** los cambios de la Ronda 1; solo procesa lo nuevo, sin importar cuántas veces se ejecute.
6. **Comparado con un enfoque tradicional** (recalcular o comparar la tabla Bronze completa contra Silver en cada ejecución), CDF evita escanear datos no modificados — el costo de cada corrida es proporcional al volumen de cambios, no al tamaño total de la tabla.
7. **Limitación a recordar:** la disponibilidad del CDF para versiones antiguas depende de que los archivos físicos no hayan sido eliminados por `VACUUM`. La política de retención del pipeline debe ser consistente con la frecuencia de ejecución.

In [0]:
# LIMPIEZA — descomentar para restablecer el ambiente
# spark.sql(f"DROP TABLE IF EXISTS {TABLA_BRONZE}")
# spark.sql(f"DROP TABLE IF EXISTS {TABLA_SILVER}")
# spark.sql(f"DROP TABLE IF EXISTS {CATALOGO}.{SCHEMA_BRONZE}.pipeline_watermarks")
# print("Limpieza completada")